# AML Signal Escalation — Team 264FC1A2

Этот notebook **воспроизводит весь пайплайн с нуля**: установка зависимостей → загрузка реальных данных из `data/raw/` → фичи → CV → модели/ансамбль → `outputs/submission.csv`.
Последняя ячейка проверяет файл на соответствие официальным правилам сдачи (ровно 6000 строк, id из test, без дублей/пропусков, `ehtimollik` в [0, 1]).

Требуется, чтобы рядом с notebook лежали `requirements.txt`, `configs/`, `src/`, `run_pipeline.py` и данные в `data/raw/` — это уже так, если notebook распакован вместе со всем проектом (`aml-escalation-hackathon-adapted.zip`).

In [ ]:
# 1. Зависимости (в Colab/любом окружении с интернетом ставятся напрямую)
%pip install -q -r requirements.txt

In [ ]:
# 2. Полный прогон пайплайна на реальных данных конкурса.
# Эквивалент `python run_pipeline.py` из терминала, но внутри notebook, чтобы весь процесс был виден и воспроизводим.
!python run_pipeline.py

In [ ]:
# 3. Загружаем реальный результат пайплайна
import pandas as pd

predictions = pd.read_csv("outputs/submission.csv")
print(f"Loaded {len(predictions)} predictions")
predictions.head()

In [ ]:
# 4. Проверка на соответствие официальным правилам сдачи хакатона
test_signals = pd.read_csv("data/raw/test_signals.csv")
test_ids = set(test_signals["signal_id"].astype(str))
pred_ids = predictions["signal_id"].astype(str)

checks = {
    "ровно 2 колонки, в порядке signal_id, ehtimollik": list(predictions.columns) == ["signal_id", "ehtimollik"],
    "ровно одна строка на каждый test signal_id": len(predictions) == len(test_ids),
    "нет дублей id": pred_ids.is_unique,
    "все id есть в test_signals.csv (нет посторонних)": set(pred_ids) <= test_ids,
    "нет пропущенных id (все id теста покрыты)": test_ids <= set(pred_ids),
    "нет пропущенных предсказаний (NaN)": predictions["ehtimollik"].notna().all(),
    "0 <= ehtimollik <= 1 для всех строк": predictions["ehtimollik"].between(0, 1).all(),
}

for name, ok in checks.items():
    print(("PASS" if ok else "FAIL"), "-", name)

assert all(checks.values()), "submission не проходит официальные правила — сдавать нельзя, см. FAIL выше"
print("\nВсе проверки пройдены. Файл готов к сдаче как team_264FC1A2.csv")

In [ ]:
# 5. Финальное имя файла под конкретную команду
import shutil

shutil.copy("outputs/submission.csv", "team_264FC1A2.csv")
print("Сохранено: team_264FC1A2.csv")